In [1]:
import json

In [2]:
with open("../data/external/all_examples_og_prompt_with_position_info.json") as f:
    data = json.load(f)

In [3]:
data[0]

{'source_file': 'Benchmark Questions Verification V2.ipynb',
 'task_id': 2,
 'prompt': 'Write a function to find the shared elements from the given two lists.',
 'code': 'def similar_elements(test_tup1, test_tup2):\n  res = tuple(set(test_tup1) & set(test_tup2))\n  return (res) ',
 'test_imports': [],
 'test_list': ['assert set(similar_elements((3, 4, 5, 6),(5, 7, 4, 10))) == set((4, 5))',
  'assert set(similar_elements((1, 2, 3, 4),(5, 4, 3, 7))) == set((3, 4))',
  'assert set(similar_elements((11, 12, 14, 13),(17, 15, 14, 13))) == set((13, 14))'],
 'instruct_code': 'def similar_elements(list1, list2):\n    return set(list1).intersection(set(list2))\n',
 'model_output': 'def similar_elements(list1, list2):\n    return set(list1).intersection(list2)\n',
 'position_info': {'base_token_pos': 189,
  'base_token_enc': 1889,
  'base_token_dec': ' list',
  'instruct_token_pos': 207,
  'instruct_token_enc': 881,
  'instruct_token_dec': 'set'}}

In [7]:
import multiprocessing as mp, psutil

def _worker(imports: str, code: str, tests: list[str], queue):
    """
    Child process: exec supplied code, then run tests.
    Sends (passed, error_msg) back through Queue.
    """
    try:
        ns = {}
        for imp in imports:
            exec(imp, ns)

        exec(code, ns)

        for test in tests:
            # Each test is an assert statement string, e.g.
            # "assert add(1, 2) == 3"
            exec(test, ns)
        queue.put((True, ""))
    except Exception as e:
        queue.put((False, repr(e)))

def run_tests(imports: str, code: str, tests: list[str], timeout=15) -> bool:
    """
    Returns True if *all* tests pass within the timeout.
    """
    q = mp.Queue()
    p = mp.Process(target=_worker, args=(imports, code, tests, q))
    p.start()
    p.join(timeout)
    if p.is_alive():
        # kill runaway child (and its children)
        proc = psutil.Process(p.pid)
        for ch in proc.children(recursive=True):
            ch.kill()
        proc.kill()
        p.join()
        return False
    passed, _ = q.get() if not q.empty() else (False, "no-result")
    return passed

In [8]:
sample = data[0]
run_tests(
    imports = sample["test_imports"],
    code = sample["instruct_code"],
    tests = sample["test_list"]
)

True

In [9]:
### need to import as well.
run_tests(
    imports = sample["test_imports"],
    code = sample["model_output"],
    tests = sample["test_list"]
)

True

In [14]:
for entry in data:
   base_output, instruct_output, test_list, imports = entry["model_output"], entry["instruct_code"], entry["test_list"], entry["test_imports"]
   instruct_pass = False
   base_pass = False
   if run_tests(
      imports = imports,
      code = instruct_output,
      tests = test_list
   ):
        instruct_pass = True
   if run_tests(
       imports  = imports,
       code = base_output,
       tests = test_list
   ):
         base_pass = True
    
   entry["instruct_pass"] = instruct_pass
   entry["base_pass"] = base_pass

In [27]:
with open("../data/external/all_examples_og_prompt_with_position_info_and_success.json","w") as f:
    json.dump(data, f, indent = 2)

In [19]:
base_pass = 0
instruct_pass = 0
instruct_pass_base_fail = 0
base_pass_instruct_fail = 0
base_pass_instruct_fail_entries = []

for entry in data:
    if entry["base_pass"]:
        base_pass += 1
    if entry["instruct_pass"]:
        instruct_pass += 1
    if entry["base_pass"] and not entry["instruct_pass"]:
        base_pass_instruct_fail += 1
        # print("Base pass but instruct fail here")
        # print(entry)
        base_pass_instruct_fail_entries.append(entry)
    if entry["instruct_pass"] and not entry["base_pass"]:
        instruct_pass_base_fail += 1
        # print("Instruct pass but base fail here")
        # print(entry)

tot = len(data)
print(f"Base pass success rate, {base_pass / tot:.2f}")
print(f"Instruct pass success rate, {instruct_pass / tot:.2f}")

Base pass success rate, 0.34
Instruct pass success rate, 0.52


In [20]:
base_pass_instruct_fail

17

In [21]:
instruct_pass_base_fail

86

In [22]:
base_pass_instruct_fail_entries[0]

{'source_file': 'Benchmark Questions Verification V2.ipynb',
 'task_id': 11,
 'prompt': 'Write a python function to remove first and last occurrence of a given character from the string.',
 'code': 'def remove_Occ(s,ch): \n    for i in range(len(s)): \n        if (s[i] == ch): \n            s = s[0 : i] + s[i + 1:] \n            break\n    for i in range(len(s) - 1,-1,-1):  \n        if (s[i] == ch): \n            s = s[0 : i] + s[i + 1:] \n            break\n    return s ',
 'test_imports': [],
 'test_list': ['assert remove_Occ("hello","l") == "heo"',
  'assert remove_Occ("abcda","a") == "bcd"',
  'assert remove_Occ("PHP","P") == "H"'],
 'instruct_code': 'def remove_Occ(string, char):\n    if string == "":\n        return ""\n    if char in string:\n        return string[:string.index(char)] + string[string.index(char)+1:]\n    else:\n        return string\n',
 'model_output': 'def remove_Occ(string, char):\n    return string.replace(char, "")\n',
 'position_info': {'base_token_pos': 

In [26]:
base_pass_instruct_fail_entries[3]

{'source_file': 'Benchmark Questions Verification V2.ipynb',
 'task_id': 223,
 'prompt': 'Write a function that takes in a sorted array, its length (n), and an element and returns whether the element is the majority element in the given sorted array. (The majority element is the element that occurs more than n/2 times.)',
 'code': 'def is_majority(arr, n, x):\n\ti = binary_search(arr, 0, n-1, x)\n\tif i == -1:\n\t\treturn False\n\tif ((i + n//2) <= (n -1)) and arr[i + n//2] == x:\n\t\treturn True\n\telse:\n\t\treturn False\ndef binary_search(arr, low, high, x):\n\tif high >= low:\n\t\tmid = (low + high)//2 \n\t\tif (mid == 0 or x > arr[mid-1]) and (arr[mid] == x):\n\t\t\treturn mid\n\t\telif x > arr[mid]:\n\t\t\treturn binary_search(arr, (mid + 1), high, x)\n\t\telse:\n\t\t\treturn binary_search(arr, low, (mid -1), x)\n\treturn -1',
 'test_imports': [],
 'test_list': ['assert is_majority([1, 2, 3, 3, 3, 3, 10], 7, 3) == True',
  'assert is_majority([1, 1, 2, 4, 4, 4, 6, 6], 8, 4) == Fa

In [1]:
### look at base and instruct 
from datasets import load_dataset

ds = load_dataset("evalplus/mbppplus")

In [2]:
df = ds['test'].to_pandas()

In [3]:
df.head()

,task_id,code,prompt,source_file,test_imports,test_list,test
0,2,"\ndef similar_elements(test_tup1, test_tup2):\...",Write a function to find the shared elements f...,Benchmark Questions Verification V2.ipynb,[],"[assert set(similar_elements((3, 4, 5, 6),(5, ...",import numpy as np\nfrom math import inf\n\nde...
1,3,\nimport math\ndef is_not_prime(n):\n if n ...,Write a python function to identify non-prime ...,Benchmark Questions Verification V2.ipynb,[],"[assert is_not_prime(2) == False, assert is_no...",import numpy as np\nfrom math import inf\n\nde...
2,4,\nimport heapq as hq\ndef heap_queue_largest(n...,Write a function to find the n largest integer...,Benchmark Questions Verification V2.ipynb,[],"[assert heap_queue_largest( [25, 35, 22, 85, 1...",import numpy as np\nfrom math import inf\n\nde...
3,6,\ndef is_Power_Of_Two(x: int): \n return x ...,Write a python function to check whether the t...,Benchmark Questions Verification V2.ipynb,[],"[assert differ_At_One_Bit_Pos(13,9) == True, a...",import numpy as np\nfrom math import inf\n\nde...
4,7,\nimport re\ndef find_char_long(text):\n retu...,Write a function to find all words which are a...,Benchmark Questions Verification V2.ipynb,[],[assert set(find_char_long('Please move back t...,import numpy as np\nfrom math import inf\n\nde...


In [4]:
def _retrieve_test(task_id):
    try:
        return df[df["task_id"] == task_id].iloc[0]["test"]
    except:
        return None

In [5]:
print(_retrieve_test(2))

import numpy as np
from math import inf

def is_floats(x) -> bool:
    # check if it is float; List[float]; Tuple[float]
    if isinstance(x, float):
        return True
    if isinstance(x, (list, tuple)):
        return all(isinstance(i, float) for i in x)
    if isinstance(x, np.ndarray):
        return x.dtype == np.float64 or x.dtype == np.float32
    return False


def assertion(out, exp, atol):
    if atol == 0 and is_floats(exp):
        atol = 1e-6
    out = set(out)
    exp = set(exp)
    if out != exp and atol != 0:
        assert np.allclose(out, exp, rtol=1e-07, atol=atol)
    else:
        assert out == exp, f"out: {out}, exp: {exp}"


inputs = [[(3, 4, 5, 6), (5, 7, 4, 10)], [(1, 2, 3, 4), (5, 4, 3, 7)], [(11, 12, 14, 13), (17, 15, 14, 13)], [(), ()], [(1, 2, 3), ()], [(), (4, 5, 6)], [(1, 2, 3, 4, 5, 6, 7, 8, 9, 10), (11, 12, 13, 14, 15, 16, 17, 18, 19, 20)], [(1, 2, 2, 3, 3, 4, 4, 5, 5), (5, 5, 6, 6, 7, 7, 8, 8, 9, 9)], [(100, 200, 300, 400, 500), (100, 200, 400, 500)]

In [6]:
import json

with open("../data/external/all_examples_og_prompt_with_position_info_and_success.json", "r") as f:
    data = json.load(f)

In [7]:
data[0]

{'source_file': 'Benchmark Questions Verification V2.ipynb',
 'task_id': 2,
 'prompt': 'Write a function to find the shared elements from the given two lists.',
 'code': 'def similar_elements(test_tup1, test_tup2):\n  res = tuple(set(test_tup1) & set(test_tup2))\n  return (res) ',
 'test_imports': [],
 'test_list': ['assert set(similar_elements((3, 4, 5, 6),(5, 7, 4, 10))) == set((4, 5))',
  'assert set(similar_elements((1, 2, 3, 4),(5, 4, 3, 7))) == set((3, 4))',
  'assert set(similar_elements((11, 12, 14, 13),(17, 15, 14, 13))) == set((13, 14))'],
 'instruct_code': 'def similar_elements(list1, list2):\n    return set(list1).intersection(set(list2))\n',
 'model_output': 'def similar_elements(list1, list2):\n    return set(list1).intersection(list2)\n',
 'position_info': {'base_token_pos': 189,
  'base_token_enc': 1889,
  'base_token_dec': ' list',
  'instruct_token_pos': 207,
  'instruct_token_enc': 881,
  'instruct_token_dec': 'set'},
 'instruct_pass': True,
 'base_pass': True}

In [8]:
import multiprocessing as mp, psutil

def _verdict_worker(imports: list[str], code: str, test: str, queue):
    """
    Child process: exec supplied code, then run tests.
    Sends (passed, error_msg) back through Queue.
    """
    try:
        ns = {}
        for imp in imports:
            exec(imp, ns)
        exec(code, ns)
        exec(test, ns)
        queue.put((True, ""))
    except Exception as e:
        queue.put((False, repr(e)))

def get_new_verdict(imports: list[str], code: str, test: str, timeout = 15) -> bool:  
    """
        Return true if *all* tests in `test` pass within the timeout.
    """  
    q = mp.Queue()
    p = mp.Process(target=_verdict_worker, args=(imports, code, test, q))
    p.start()
    p.join(timeout)
    if p.is_alive():
        # kill runaway child (and its children)
        proc = psutil.Process(p.pid)
        for ch in proc.children(recursive=True):
            ch.kill()
        proc.kill()
        p.join()
        return False
    passed, _ = q.get() if not q.empty() else (False, "no-result")
    return passed

In [9]:
def _verify_info_success_more_riguor(entry, verbose = False, **log_dict):
    test = _retrieve_test(entry["task_id"])
    if test is not None:
        instruct_verdict = get_new_verdict(entry["test_imports"], entry["instruct_code"], test)
        base_verdict = get_new_verdict(entry["test_imports"], entry["model_output"], test)
        if verbose:
            print("New base verdict", base_verdict)
            print("New instruct verdict", instruct_verdict)
        if entry["instruct_pass"] != instruct_verdict:
            print(f"Verdict changed for instruct for {log_dict}")
        if entry["base_pass"] != base_verdict:
            print(f"Verdict changed for base for {log_dict}")
        entry["instruct_pass"] = instruct_verdict
        entry["base_pass"] = base_verdict
    else:
        print(f"Did not find tests for {log_dict}")

In [10]:
data[0]

{'source_file': 'Benchmark Questions Verification V2.ipynb',
 'task_id': 2,
 'prompt': 'Write a function to find the shared elements from the given two lists.',
 'code': 'def similar_elements(test_tup1, test_tup2):\n  res = tuple(set(test_tup1) & set(test_tup2))\n  return (res) ',
 'test_imports': [],
 'test_list': ['assert set(similar_elements((3, 4, 5, 6),(5, 7, 4, 10))) == set((4, 5))',
  'assert set(similar_elements((1, 2, 3, 4),(5, 4, 3, 7))) == set((3, 4))',
  'assert set(similar_elements((11, 12, 14, 13),(17, 15, 14, 13))) == set((13, 14))'],
 'instruct_code': 'def similar_elements(list1, list2):\n    return set(list1).intersection(set(list2))\n',
 'model_output': 'def similar_elements(list1, list2):\n    return set(list1).intersection(list2)\n',
 'position_info': {'base_token_pos': 189,
  'base_token_enc': 1889,
  'base_token_dec': ' list',
  'instruct_token_pos': 207,
  'instruct_token_enc': 881,
  'instruct_token_dec': 'set'},
 'instruct_pass': True,
 'base_pass': True}

In [11]:
_verify_info_success_more_riguor(data[0], verbose = True, log_dict={"id": 0})

New base verdict True
New instruct verdict True


In [12]:
for idx, entry in enumerate(data):
    _verify_info_success_more_riguor(entry, verbose = False, log_dict = {"id": idx})

Verdict changed for instruct for {'log_dict': {'id': 3}}
Verdict changed for base for {'log_dict': {'id': 3}}
Verdict changed for base for {'log_dict': {'id': 6}}
Did not find tests for {'log_dict': {'id': 36}}
Verdict changed for instruct for {'log_dict': {'id': 39}}
Verdict changed for base for {'log_dict': {'id': 39}}
Verdict changed for base for {'log_dict': {'id': 46}}
Verdict changed for instruct for {'log_dict': {'id': 50}}
Verdict changed for base for {'log_dict': {'id': 50}}
Verdict changed for base for {'log_dict': {'id': 54}}
Verdict changed for instruct for {'log_dict': {'id': 60}}
Verdict changed for base for {'log_dict': {'id': 60}}
Did not find tests for {'log_dict': {'id': 61}}
Verdict changed for instruct for {'log_dict': {'id': 62}}
Verdict changed for base for {'log_dict': {'id': 62}}
Did not find tests for {'log_dict': {'id': 63}}
Verdict changed for instruct for {'log_dict': {'id': 64}}
Verdict changed for base for {'log_dict': {'id': 64}}
Verdict changed for instr

In [20]:
data[384]

{'source_file': 'Benchmark Questions Verification V2.ipynb',
 'task_id': 802,
 'prompt': 'Write a python function to count the number of rotations required to generate a sorted array. https://www.geeksforgeeks.org/count-of-rotations-required-to-generate-a-sorted-array/',
 'code': 'def count_rotation(arr):   \n    for i in range (1,len(arr)): \n        if (arr[i] < arr[i - 1]): \n            return i  \n    return 0',
 'test_imports': [],
 'test_list': ['assert count_rotation([3,2,1]) == 1',
  'assert count_rotation([4,5,1,2,3]) == 2',
  'assert count_rotation([7,8,9,1,2,3]) == 3',
  'assert count_rotation([1,2,3]) == 0',
  'assert count_rotation([1,3,2]) == 2'],
 'instruct_code': 'def count_rotation(arr):\n    n = len(arr)\n    left = 0\n    right = n - 1\n    while left < right:\n        if arr[left] > arr[right]:\n            return left\n        left += 1\n    return right + 1\n',
 'model_output': 'def count_rotation(arr):\n    # Write your code here\n    return 0\n',
 'position_inf

In [21]:
with open("../data/external/all_examples_og_prompt_with_position_info_and_success_V2.json", "w") as f:
    json.dump(data, f, indent = 2)

In [23]:
base_pass = 0
instruct_pass = 0
instruct_pass_base_fail = 0
base_pass_instruct_fail = 0
base_pass_instruct_fail_entries = []

for entry in data:
    if entry["base_pass"]:
        base_pass += 1
    if entry["instruct_pass"]:
        instruct_pass += 1
    if entry["base_pass"] and not entry["instruct_pass"]:
        base_pass_instruct_fail += 1
        # print("Base pass but instruct fail here")
        # print(entry)
        base_pass_instruct_fail_entries.append(entry)
    if entry["instruct_pass"] and not entry["base_pass"]:
        instruct_pass_base_fail += 1
        # print("Instruct pass but base fail here")
        # print(entry)

tot = len(data)
print(f"Base pass success rate, {base_pass / tot:.2f}")
print(f"Instruct pass success rate, {instruct_pass / tot:.2f}")

Base pass success rate, 0.28
Instruct pass success rate, 0.47
